# 09 — DPO from scratch: deriving the loss

**Phase 8, Part C.** This notebook derives `llmlab/train/dpo.py`'s loss from the RLHF problem
statement, the way the spec asks: work out the ~5 lines of math first, then show the ~20 lines of
code that fall out of it are *exactly* what's running in `DPOTrainer`. Companion code: `dpo.py`
(loss), `dpo_loader.py` (paired data), `dpo_trainer.py` (the training loop). Results from the
real run land in the last section once `scripts/dpo.py` has been run.


## 1. The RLHF objective (what we're actually trying to solve)

Classic RLHF (InstructGPT-style) trains a reward model $r(x, y)$ on human preference pairs, then
optimizes the policy $\pi_\theta$ against it with a KL penalty back to a reference policy
$\pi_{\mathrm{ref}}$ (the SFT model — Part A's `p8_sft-s-dictionary`), so the policy can't drift
arbitrarily far chasing reward:

$$
\max_{\pi_\theta} \; \mathbb{E}_{x \sim D,\, y \sim \pi_\theta(\cdot|x)}\big[r(x, y)\big]
\;-\; \beta \, \mathbb{E}_{x \sim D}\Big[\mathrm{KL}\big(\pi_\theta(\cdot|x) \,\|\, \pi_{\mathrm{ref}}(\cdot|x)\big)\Big]
$$

$\beta$ sets the exchange rate between reward and staying close to the reference. This is
normally optimized with PPO — a genuinely separate RL loop, on top of a genuinely separate reward
model that itself needed training first. DPO's whole contribution is noticing you don't need
either piece.


## 2. Step 1 — the optimal policy for ANY fixed reward has a closed form

Fix $x$ and treat the inner maximization over $\pi_\theta(\cdot|x)$ as a per-prompt problem.
Writing the KL term out and flipping the sign (turning the max into a min of a free-energy-style
functional):

$$
\min_{\pi_\theta} \; \mathbb{E}_{y \sim \pi_\theta}\Big[ \log\frac{\pi_\theta(y|x)}{\pi_{\mathrm{ref}}(y|x)} - \frac{1}{\beta} r(x,y) \Big]
$$

This is minimized (a standard result — the unique minimizer of $\mathbb{E}_\pi[\log(\pi/q)]$ for a
fixed "tilted" target $q$ is $\pi = q$ itself, by Gibbs' inequality / non-negativity of KL)
by:

$$
\pi^*(y|x) = \frac{1}{Z(x)}\, \pi_{\mathrm{ref}}(y|x) \, \exp\!\Big(\frac{1}{\beta} r(x,y)\Big),
\qquad Z(x) = \sum_y \pi_{\mathrm{ref}}(y|x) \exp\!\Big(\frac{1}{\beta} r(x,y)\Big)

$$

Read this as: start from the reference distribution, reweight every response by how much reward
it gets (exponentially, scaled by $1/\beta$), renormalize. $Z(x)$ is just the normalizer — an
intractable sum over every possible response, which is exactly why nobody computes $\pi^*$
directly. DPO's next move is to never need to.


## 3. Step 2 — invert it: express the reward in terms of the policy

Solve the Step-1 equation for $r(x,y)$ instead of $\pi^*$:

$$
r(x,y) = \beta \log\frac{\pi^*(y|x)}{\pi_{\mathrm{ref}}(y|x)} + \beta \log Z(x)
$$

This is the algebraic pivot the whole method rests on. If $\pi_\theta$ IS the optimal policy for
some underlying reward, then that reward is recoverable from the policy's log-ratio to the
reference — up to a per-prompt constant $\beta \log Z(x)$ that doesn't depend on $y$ at all.
That "doesn't depend on $y$" clause is about to matter a lot.


## 4. Step 3 — plug into Bradley-Terry, watch $Z(x)$ cancel

Preference data doesn't give you $r(x,y)$ directly, only *which of two responses was preferred*.
The standard model for that is Bradley-Terry: given chosen $y_w$ and rejected $y_l$ for the same
prompt $x$,

$$
P(y_w \succ y_l \mid x) = \sigma\big(r(x,y_w) - r(x,y_l)\big)
$$

Substitute Step 2's expression for $r$ on both sides. Both $y_w$ and $y_l$ share the same $x$, so
the SAME $\beta \log Z(x)$ term appears in both and cancels in the subtraction:

$$
r(x,y_w) - r(x,y_l) = \beta \log\frac{\pi_\theta(y_w|x)}{\pi_{\mathrm{ref}}(y_w|x)} - \beta \log\frac{\pi_\theta(y_l|x)}{\pi_{\mathrm{ref}}(y_l|x)}
$$

**This is the payoff.** The intractable normalizer is gone. The preference probability is now
written *purely* in terms of the policy and the reference — two things we can already
compute a forward pass through. No reward model was ever built; $r$ was only ever a symbol we
used to get here and then eliminated.


## 5. Step 4 — maximum likelihood over the preference dataset = the DPO loss

Given the substituted Bradley-Terry probability, maximize the log-likelihood of the observed
preferences (equivalently, minimize the negative log-likelihood) over a dataset of triples
$(x, y_w, y_l)$:

$$
\mathcal{L}_{\mathrm{DPO}}(\theta) = -\,\mathbb{E}_{(x,y_w,y_l)}\Big[\log \sigma\Big(
\beta \log\frac{\pi_\theta(y_w|x)}{\pi_{\mathrm{ref}}(y_w|x)}
- \beta \log\frac{\pi_\theta(y_l|x)}{\pi_{\mathrm{ref}}(y_l|x)}
\Big)\Big]
$$

That's the whole derivation: a KL-regularized RL objective, solved in closed form, substituted
into a preference model, turned into a supervised loss. `llmlab/train/dpo.py::dpo_loss` is this
equation and nothing else — `sequence_logprobs` computes each $\log \pi(y|x)$ (summed token
log-probs over the assistant span, using Part A's exact loss mask), and `dpo_loss` combines four
of them into the scalar above.


In [1]:
import math
import sys
from pathlib import Path

import torch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from llmlab.train.dpo import dpo_loss, sequence_logprobs

print("dpo.py loaded from", (ROOT / "src/llmlab/train/dpo.py").resolve())


dpo.py loaded from /Users/adityaram/2026/llm/src/llmlab/train/dpo.py


## 6. Sanity-check the code against the math with toy numbers

Two checks anyone implementing this should run before trusting a real training loop:

1. **At initialization**, the policy IS the reference (DPO always starts both from the same SFT
   checkpoint), so every log-ratio is exactly 0 — the loss must equal $-\log\sigma(0) = \log 2
   \approx 0.693$, regardless of $\beta$ or of what the actual chosen/rejected text is.
2. **Sensitivity to $\beta$**: for a FIXED log-ratio gap, larger $\beta$ should make the loss
   fall faster as training pushes the policy toward preferring chosen — $\beta$ literally
   multiplies the logit fed to the sigmoid.


In [2]:
# Check 1: policy == reference -> loss == log(2), independent of beta.
zeros = torch.zeros(8)
for beta in (0.05, 0.1, 0.5, 1.0):
    loss, m = dpo_loss(zeros, zeros, zeros, zeros, beta=beta)
    assert abs(loss.item() - math.log(2)) < 1e-6, (beta, loss.item())
print(f"loss at zero drift == log(2) == {math.log(2):.4f}  (confirmed for beta in 0.05..1.0)")

# Check 2: same log-ratio gap, larger beta -> lower loss (steeper reward for the same evidence).
gap = torch.tensor([2.0])   # policy_chosen_lp - ref_chosen_lp, rejected side held at the reference
zero = torch.tensor([0.0])
print(f"{'beta':>6} {'loss':>10} {'reward_margin':>15}")
for beta in (0.01, 0.1, 0.5, 1.0):
    loss, m = dpo_loss(gap, zero, zero, zero, beta=beta)
    print(f"{beta:>6} {loss.item():>10.4f} {m['reward_margin']:>15.4f}")


loss at zero drift == log(2) == 0.6931  (confirmed for beta in 0.05..1.0)
  beta       loss   reward_margin
  0.01     0.6832          0.0200
   0.1     0.5981          0.2000
   0.5     0.3133          1.0000
   1.0     0.1269          2.0000


Both hold: the loss is pinned at $\log 2$ with zero drift no matter $\beta$
(check 1), and for identical evidence a larger $\beta$ turns the same log-ratio gap into a larger
reward margin and a lower loss (check 2) — $\beta$ is doing exactly the "how sharply do we punish
preference violations" job the derivation predicts. A LARGER $\beta$ also means the KL penalty in
the original RLHF objective was effectively WEAKER (re-read Section 1: $\beta$ multiplies the KL
term there, and ends up dividing it inside $\exp(r/\beta)$ in Section 2) — so a bigger $\beta$
lets the policy drift further from the reference for the same reward gain. That's the practical
knob: small $\beta$ = conservative/stays close to SFT, large $\beta$ = aggressive/bigger updates.


## 7. A real preference triple, end to end

Load the real 16k tokenizer and a tiny 2-layer GPT (same fixture shape as `tests/test_dpo.py`),
run one real triple from `data/dpo/dictionary_pairs/train.jsonl` through `sequence_logprobs` for
both a "policy" and a frozen "reference" copy, and confirm the assembled loss/metrics match what
`DPOTrainer._step_loss` computes internally.


In [3]:
import json

from tokenizers import Tokenizer

from llmlab.data.dpo_loader import DPODataset
from llmlab.model import GPT, ModelConfig

tokenizer = Tokenizer.from_file(str(ROOT / "data/tokenized/tokenizers/hf_bpe_16k/tokenizer.json"))
train_path = ROOT / "data/dpo/dictionary_pairs/train.jsonl"
rows = [json.loads(l) for l in train_path.read_text().splitlines()[:1]]
print(json.dumps(rows[0], indent=2))

tiny_cfg = ModelConfig(vocab_size=16000, d_model=32, n_layers=2, n_heads=2, n_kv_heads=2, head_dim=16, max_seq_len=512)
policy = GPT(tiny_cfg)
reference = GPT(tiny_cfg)
reference.load_state_dict(policy.state_dict())  # identical at "init", like a fresh DPO run
reference.eval()
for p in reference.parameters():
    p.requires_grad_(False)

ds = DPODataset(rows, tokenizer, max_len=384)
x_c, y_c, x_r, y_r = ds.collate([0], torch.device("cpu"))

policy_c, policy_r = sequence_logprobs(policy, x_c, y_c), sequence_logprobs(policy, x_r, y_r)
with torch.no_grad():
    ref_c, ref_r = sequence_logprobs(reference, x_c, y_c), sequence_logprobs(reference, x_r, y_r)
loss, metrics = dpo_loss(policy_c, policy_r, ref_c, ref_r, beta=0.1)

print(f"\nloss = {loss.item():.4f}  (expect log(2) = {math.log(2):.4f}, since policy==reference here)")
print(metrics)


{
  "instruction": "What is 'atar'?",
  "chosen": "Atar is a special, sweet-smelling oil or perfume that people make from flowers.",
  "rejected": "Atar is not a word I recognize, but it reminds me of that time I tried to find my way through a dense forest and stumbled upon a clearing. The flowers there had a fragrance unlike any other, and I wondered what name they might have. That's the thing about words, they can lead you on a journey, but not always to the destination you wanted.",
  "meta": {
    "word": "atar",
    "failure_mode": "off_format"
  }
}

loss = 0.6931  (expect log(2) = 0.6931, since policy==reference here)
{'reward_chosen': 0.0, 'reward_rejected': 0.0, 'reward_margin': 0.0, 'reward_accuracy': 0.0, 'kl_chosen': 0.0, 'kl_rejected': 0.0}


## 8. What training actually watches, and why

`DPOTrainer` doesn't log a "does the loss go down" chart alone — for this problem that's a weak
signal (the loss going down could mean the policy is learning real preferences, OR just
inflating $\log\pi(y_w)$ everywhere regardless of $y_l$, a known DPO failure mode). Three
numbers, all computed by `dpo_loss`, tell the difference:

- **`reward_margin`** = mean($r_{\mathrm{chosen}} - r_{\mathrm{rejected}}$) — should rise. This is
  the direct target of the loss.
- **`reward_accuracy`** = fraction of pairs where the policy ranks chosen above rejected — the
  preference-pair analogue of classification accuracy; should approach 1.0 if the policy is
  genuinely learning the WRONG/verbose/off_format failure modes are worse, not just noise.
- **`kl_chosen` / `kl_rejected`** = mean($\log\pi_\theta(y|x) - \log\pi_{\mathrm{ref}}(y|x)$) per
  side — a diagnostic for drift. If this grows large while `pretrain_val_ppl` (the forgetting
  probe carried over from Part A) also climbs, the policy is drifting away from BOTH the
  reference's preferences AND its general language ability — the over-optimization DPO papers
  warn about, and the reason `epochs: 1` and a `lr` ~4x lower than SFT's are this project's
  starting defaults (`configs/dpo_s_dictionary.yaml`).


## 9. Results from the real run

Run `20260719_p8_dpo-s-dictionary` (policy + frozen reference both start from Part A's full-FT
SFT model). Full narrative + honest caveats: `docs/results/finetune_report.md` Part C.

**Stopped early, on purpose, at step 91/172 (53% of one epoch).** Per-step wall-clock degraded
steeply over the run (ruled out batch-shape as the cause — checked directly), most likely a
sustained-heavy-MPS-workload effect (DPO runs FOUR forward passes/step vs SFT's one). Meanwhile
the val signal had already saturated: by step 75, reward_accuracy 79%->95.8%, reward_margin
0.04->8.97 (beta=0.1 -> an average policy/reference log-ratio gap of ~90 nats in under half an
epoch). Continuing mostly deepens an already-large divergence; interrupting cleanly (SIGINT,
which `DPOTrainer` catches and checkpoints) beat letting an increasingly-slow run balloon toward
CLAUDE.md's "multi-hour, ask first" territory.

| metric | SFT (pre-DPO) | DPO | read |
|---|---:|---:|---|
| reward_accuracy vs frozen SFT ref | 0.0%¹ | **95.8%** | real preference learning |
| reward_margin vs frozen SFT ref | 0.0000 | **8.9694** | large, real drift |
| stop-rate | 82% | **99%** | genuinely length-unconfounded win |
| mean answer length (tok) | 33.9 | 16.5 | see the catch below |
| pretrain val ppl (forgetting) | 40.10 | 44.82 | +11.8% further, in 91 steps |

¹ degenerate tie (policy==reference -> every log-ratio is 0), not evidence SFT preferred rejected.

**The honest catch.** A reference-FREE check (does a model, alone, already give chosen a higher
raw summed log-prob than rejected?) says yes 95.8% of the time -- for BOTH SFT and DPO, gap
+624->+713 nats. That is a **length confound**, not preference understanding: chosen averages
~59 tokens, rejected averages ~461 (the verbose failure mode dominates), and an un-normalized
summed log-prob is mechanically larger for a shorter sequence almost regardless of content.
Section 4/5's derivation shows exactly why DPO's own reward (relative to the reference, on the
SAME `y`) is immune to this -- but the training GRADIENT is not, and the answer-length collapse
(34->16.5 tokens) plus terse/vague qualitative samples (`experiments/.../samples/step_000050.txt`,
e.g. "What is a go-between?" -> "A wit.") show the model partly exploiting "shorter closes the
gap" alongside genuinely learning to stop rambling. A well-documented real RLHF/DPO failure mode
(verbosity/length bias), not a bug in this implementation -- the fix for a follow-up run is
length-balancing the rejected generation or length-normalizing the loss (`logp / len`, a
documented DPO variant), not re-deriving anything above.
